# Molecular atmosphere testing

This notebook tests the backend setup for a molecular atmosphere.

In [ ]:
import cases
import eradiate
import matplotlib.pyplot as plt
import seaborn as sns
from eradiate.contexts import KernelContext
from util import Result, reshape_pplane

import eradiate_disort as ed

eradiate.set_mode("ckd")
sns.set_theme(style="ticks")

SPP = 100_000
if eradiate.get_mode().is_ckd:
    SPP //= 16

exp = cases.molecular(sza=30.0)
ctx = KernelContext()
exp.atmosphere.eval_radprops(ctx.si, optional_fields=True)

In [ ]:
results = {}
_cases = {
    "rayleigh": {
        "has_absorption": False,
        "has_scattering": True,
        "surface_reflectance": 0.0,
    },
    "absorption": {
        "has_absorption": True,
        "has_scattering": False,
        "surface_reflectance": 1.0,
    },
    "full_black": {
        "has_absorption": True,
        "has_scattering": True,
        "surface_reflectance": 0.0,
    },
    "full_white": {
        "has_absorption": True,
        "has_scattering": True,
        "surface_reflectance": 1.0,
    },
}

for case_id, kwargs in _cases.items():
    print(f"Processing case {case_id!r}")
    if case_id in results:
        continue

    exp = cases.molecular(**kwargs)
    result = Result()

    result.mitsuba = eradiate.run(exp, spp=SPP)["radiance"].squeeze()
    backend = ed.EradiateDisortBackend()
    result.disort = reshape_pplane(backend.run(exp))

    results[case_id] = result

In [ ]:
ncases = len(_cases)
ncols = 2
nrows = ncases // ncols + min(ncases % ncols, 1)

fig, axs = plt.subplots(
    nrows, ncols, figsize=(4 * ncols, 3 * nrows), layout="constrained", squeeze=False
)

for i, case_id in enumerate(_cases.keys()):
    irow = i // ncols
    icol = i % ncols
    ax = axs.ravel()[i]
    result = results[case_id]

    ax.plot(result.mitsuba["vza"], result.mitsuba, label="Mitsuba" if i == 0 else None)
    ax.plot(
        result.disort["vza"],
        result.disort,
        label="CDISORT" if i == 0 else None,
        ls="--",
    )

    ax.set_title(case_id)
    ax.set_xlabel("θ [°]")
    if icol == 0:
        ax.set_ylabel("Radiance [W/m²/sr]")

else:
    while (i := i + 1) < axs.size:
        ax = axs.ravel()[i]
        ax.set_axis_off()

fig.legend(title="Backend", ncol=2, loc="outside upper center")

plt.show()